# 02b Flooding — FEMA Zones & Exposure

Run independently for the seven confirmed parcels. This notebook has no terrain dependency. It saves FEMA polygons, zone-area tables, floodplain exposure, and a flood-zone map. Verify the asset's effective date and coverage before using results. `SFHA_TF` and `FLD_ZONE` are required. A missing layer or schema stops processing rather than becoming zero flood exposure.

Floodway is reported only when the asset contains a recognizable floodway designation. FEMA mapping is not a prediction of all flooding, erosion, or trail conditions.
Maps distinguish mapped floodways with diagonal hatching and separate zone X subtypes from `ZONE_SUBTY`. A parcel close-up, detailed polygons, and subtype area summaries are exported. These details depend on the source fields; missing subtypes are not guessed.


Run cells in order. Settings, helper functions, calculations, maps, and exports are separated. Figures are PNG only. Install dependencies once if needed: `earthengine-api geopandas matplotlib numpy pandas shapely pyogrio`.

## 1. Imports

Only libraries used by this analysis are loaded.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from shapely import force_2d
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


## 2. Settings and local paths

Keep data sources, parcel IDs, and output locations in one place.

In [ ]:
EE_PROJECT = "stalwart-glider-279103"
METRIC_CRS = "EPSG:26914"
M_PER_MILE = 1609.344
ACRE_M2 = 4046.8564224
GOAT_ISLAND_PARCEL_IDS = [
    "65079661510030000",
    "65130905010020000",
    "65130905010030000",
    "65130905010050000",
    "65130905010060000",
    "65146334010100000",
    "65146334010140000",
]
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "Park_Parcels/Park_Parcles.shp").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside goat-island-analysis.")
PARCEL_PATH = PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp"
OUTPUT_DIR = PROJECT_ROOT / "goat_island_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEMA_ASSET = f"projects/{EE_PROJECT}/assets/nfhl_flood_zones"
FEMA_EFFECTIVE_DATE = ""  # Confirm against the source metadata.
FLOOD_DIR = OUTPUT_DIR / "flooding"
FLOOD_DIR.mkdir(exist_ok=True)


## 3. Earth Engine connection

Reuse saved authentication when available.

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready. Outputs:", OUTPUT_DIR)


## 4. Seven-parcel boundary

Validate parcel selection and geometry, then create the one-mile study area.

In [ ]:
parcels = gpd.read_file(PARCEL_PATH)
if parcels.crs is None:
    raise ValueError("Park parcels must have a defined coordinate system.")
parcel_numbers = parcels["Parcel_Num"].astype(str).str.strip()
missing_ids = set(GOAT_ISLAND_PARCEL_IDS) - set(parcel_numbers)
if missing_ids:
    raise ValueError(f"Required Goat Island parcels are missing: {sorted(missing_ids)}")
selected = parcels.loc[parcel_numbers.isin(GOAT_ISLAND_PARCEL_IDS)].copy()
if selected.empty:
    raise ValueError("No matching Goat Island project parcels.")
if selected.geometry.isna().any() or selected.geometry.is_empty.any():
    raise ValueError("Selected Goat Island parcels contain missing or empty geometry.")
selected["geometry"] = selected.geometry.map(force_2d).make_valid()
if not selected.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("Selected parcels must contain polygon geometry.")
preserve_gdf = selected.dissolve().reset_index(drop=True).to_crs("EPSG:4326")
BOUNDARY_SOURCE = (
    "Park_Parcels/Park_Parcles.shp; seven project-confirmed Parcel_Num values"
)
preserve_m = preserve_gdf.to_crs(METRIC_CRS)
study1_m = gpd.GeoDataFrame(geometry=preserve_m.buffer(M_PER_MILE), crs=METRIC_CRS)
preserve_area_acres = preserve_m.area.sum() / ACRE_M2
print(
    f"BOUNDARY: {BOUNDARY_SOURCE}; {len(selected)} parcel(s); area = {preserve_area_acres:.1f} acres"
)
preserve_gdf.to_file(OUTPUT_DIR / "preserve_boundary.geojson", driver="GeoJSON")
study1_m.to_crs(4326).to_file(
    OUTPUT_DIR / "one_mile_study_area.geojson", driver="GeoJSON"
)
roi = ee.Geometry(
    json.loads(json.dumps(preserve_gdf.geometry.iloc[0].__geo_interface__))
)
roi1 = ee.Geometry(
    json.loads(json.dumps(study1_m.to_crs(4326).geometry.iloc[0].__geo_interface__))
)


In [ ]:
BOUNDARY_HASH = hashlib.sha256(preserve_gdf.geometry.iloc[0].wkb).hexdigest()
PARCEL_IDS = sorted(GOAT_ISLAND_PARCEL_IDS)


## 5. Reusable helpers

Formatting and export helpers are defined once. Analytical rules are unchanged.

In [ ]:
def save_plot(fig, directory, name):
    """Save a PNG figure and display it in the notebook."""
    fig.savefig(directory / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def detailed_label(row):
    """Preserve FEMA zone and subtype distinctions in readable map labels."""
    zone, subtype = (row["FLD_ZONE"], row["ZONE_SUBTY"])
    if "FLOODWAY" in subtype:
        return f"{zone or 'Unspecified zone'} — mapped floodway"
    if zone == "X":
        return {
            "0.2 PCT ANNUAL CHANCE FLOOD HAZARD": "X — 0.2% annual-chance flood hazard",
            "AREA OF MINIMAL FLOOD HAZARD": "X — minimal flood hazard",
            "AREA WITH REDUCED FLOOD RISK DUE TO LEVEE": "X — reduced risk due to levee",
        }.get(subtype, "X — " + (subtype.lower() if subtype else "subtype unspecified"))
    return zone + (" — " + subtype.lower() if subtype else " — other mapped area")


def draw_detailed_flood_map(parcel_view=False):
    """Draw floodway hatching and actual zone X subtypes at either map extent."""
    fig, ax = plt.subplots(figsize=(11, 8))
    labels = sorted(
        styles,
        key=lambda label: (
            "FLOODWAY"
            in flood_detail.loc[flood_detail.map_class.eq(label), "ZONE_SUBTY"].iloc[0],
            label,
        ),
    )
    for label in labels:
        flood_detail.loc[flood_detail.map_class.eq(label)].plot(ax=ax, **styles[label])
    study1_m.boundary.plot(ax=ax, color="#666666", linestyle="--", linewidth=1)
    preserve_m.boundary.plot(ax=ax, color="black", linewidth=1.6)
    if parcel_view:
        xmin, ymin, xmax, ymax = preserve_m.total_bounds
        padding = max(xmax - xmin, ymax - ymin) * 0.04
        ax.set_xlim(xmin - padding, xmax + padding)
        ax.set_ylim(ymin - padding, ymax + padding)
    legend = [Patch(label=label, **styles[label]) for label in labels]
    legend += [
        Line2D([0], [0], color="black", linewidth=1.6, label="Seven-parcel boundary"),
        Line2D([0], [0], color="#666666", linestyle="--", label="One-mile boundary"),
    ]
    ax.legend(
        handles=legend,
        title="FEMA zone and subtype",
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        fontsize=8,
    )
    ax.set(
        title="Goat Island — "
        + (
            "parcel floodway and flood-zone detail"
            if parcel_view
            else "floodways and flood-zone subtypes, one-mile surroundings"
        ),
        xlabel="UTM Easting (m)",
        ylabel="UTM Northing (m)",
    )
    ax.ticklabel_format(style="plain", useOffset=False)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    length = 500 if parcel_view else 1000
    sx = x0 + 0.06 * (x1 - x0)
    sy = y0 + 0.05 * (y1 - y0)
    ax.plot([sx, sx + length], [sy, sy], color="black", linewidth=2)
    ax.text(
        sx + length / 2,
        sy + 0.015 * (y1 - y0),
        f"{length:g} m",
        ha="center",
        fontsize=8,
    )
    fig.text(
        0.5,
        0.02,
        "Source: configured FEMA asset • effective date: "
        + (FEMA_EFFECTIVE_DATE or "unverified")
        + "\nHatching identifies mapped floodway. Zone X categories retain their source subtypes; reduced risk does not mean no risk.",
        ha="center",
        fontsize=8,
    )
    save_plot(
        fig, FLOOD_DIR, "flood_parcel_detail" if parcel_view else "flood_zones_map"
    )


## 6. Load and validate FEMA polygons

Check required fields, repair geometry, clip to the study area, and save the source polygons.

In [ ]:
nfhl = ee.FeatureCollection(FEMA_ASSET).filterBounds(roi1)
count = nfhl.size().getInfo()
if count == 0:
    raise RuntimeError(
        "No FEMA features in the study area. Verify asset coverage; do not assume no flood hazard."
    )
properties = ee.Feature(nfhl.first()).propertyNames().getInfo()
if not {"SFHA_TF", "FLD_ZONE"}.issubset(properties):
    raise ValueError(f"FEMA schema missing required fields: {properties}")
raw = nfhl.getInfo()
flood = gpd.GeoDataFrame.from_features(raw["features"], crs=4326).to_crs(METRIC_CRS)
flood = flood.loc[flood.geometry.notna() & ~flood.geometry.is_empty].copy()
flood["geometry"] = flood.geometry.make_valid()
flood = gpd.clip(flood, study1_m)
if flood.empty:
    raise RuntimeError("No usable FEMA polygons after clipping.")
flood.to_crs(4326).to_file(FLOOD_DIR / "fema_zones.geojson", driver="GeoJSON")


## 7. Summarize parcel exposure

Export mapped coverage, flood-zone areas, SFHA exposure, and floodway acreage.

In [ ]:
sfha_local = flood.loc[flood.SFHA_TF.astype(str).str.upper().eq("T")]
parcel = preserve_m.geometry.iloc[0]
sfha_geom = sfha_local.geometry.union_all().intersection(parcel)
sfha_acres = sfha_geom.area / ACRE_M2
coverage_geom = flood.geometry.union_all().intersection(parcel)
coverage_pct = 100 * coverage_geom.area / parcel.area
records = []
for zone, features in flood.groupby("FLD_ZONE", dropna=False):
    area = features.geometry.union_all().intersection(parcel).area / ACRE_M2
    if area > 0:
        records.append(
            {
                "fema_zone": str(zone),
                "area_acres": area,
                "pct_parcel": 100 * area / preserve_area_acres,
            }
        )
table = pd.DataFrame(records)
table.to_csv(FLOOD_DIR / "flood_zone_areas.csv", index=False)
summary = {
    "parcel_acres": preserve_area_acres,
    "mapped_coverage_pct": coverage_pct,
    "sfha_acres": sfha_acres,
    "sfha_pct_parcel": 100 * sfha_acres / preserve_area_acres,
    "effective_date": FEMA_EFFECTIVE_DATE or "unverified",
}
if "ZONE_SUBTY" in flood:
    floodway = flood.loc[
        flood.ZONE_SUBTY.fillna("").str.contains("FLOODWAY", case=False)
    ]
    summary["floodway_acres"] = (
        floodway.geometry.union_all().intersection(parcel).area / ACRE_M2
    )
else:
    summary["floodway_status"] = "ZONE_SUBTY absent; not assessed"
pd.DataFrame([summary]).to_csv(FLOOD_DIR / "flood_summary.csv", index=False)


## 8. Record source metadata

Save provenance and flag incomplete coverage or an unverified effective date.

In [ ]:
metadata = {
    "boundary_hash": BOUNDARY_HASH,
    "parcel_ids": PARCEL_IDS,
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "source": FEMA_ASSET,
    "effective_date": FEMA_EFFECTIVE_DATE or "unverified",
    "mapped_coverage_pct": coverage_pct,
    "source_properties": properties,
    "rasterization_rule": "SFHA_TF = T; outside mapped FEMA coverage remains unknown",
}
(FLOOD_DIR / "manifest.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
display(pd.DataFrame([summary]))
display(table)
if coverage_pct < 99:
    print(
        "WARNING: Some parcel area lacks FEMA polygons; retain as unknown in integration."
    )
if not FEMA_EFFECTIVE_DATE:
    print(
        "WARNING: FEMA effective date is unverified. Check source metadata before reporting."
    )


## 9. Classify floodways and zone X subtypes

Use the actual FLD_ZONE and ZONE_SUBTY values. Export detailed polygons without guessing missing subtypes.

In [ ]:
flood_detail = flood.copy()
flood_detail["FLD_ZONE"] = (
    flood_detail.FLD_ZONE.fillna("").astype(str).str.strip().str.upper()
)
if "ZONE_SUBTY" not in flood_detail:
    flood_detail["ZONE_SUBTY"] = ""
    print("ZONE_SUBTY missing: floodway and X subtype distinctions cannot be inferred.")
flood_detail["ZONE_SUBTY"] = (
    flood_detail.ZONE_SUBTY.fillna("").astype(str).str.strip().str.upper()
)
flood_detail["map_class"] = flood_detail.apply(detailed_label, axis=1)
flood_detail.to_crs(4326).to_file(
    FLOOD_DIR / "fema_zones_detailed.geojson", driver="GeoJSON"
)
floodway_detail = flood_detail.loc[
    flood_detail.ZONE_SUBTY.str.contains("FLOODWAY", regex=False)
].copy()
if not floodway_detail.empty:
    floodway_detail.to_crs(4326).to_file(
        FLOOD_DIR / "mapped_floodways.geojson", driver="GeoJSON"
    )


## 10. Summarize detailed categories

Export subtype areas for the parcels and full one-mile buffer.

In [ ]:
detail_records = []
for study_label, geometry in [
    ("Seven selected parcels", preserve_m.geometry.iloc[0]),
    ("Full one-mile buffer (includes parcels)", study1_m.geometry.iloc[0]),
]:
    for (zone, subtype, label), features in flood_detail.groupby(
        ["FLD_ZONE", "ZONE_SUBTY", "map_class"], dropna=False
    ):
        area = features.geometry.union_all().intersection(geometry).area / ACRE_M2
        detail_records.append(
            {
                "study_zone": study_label,
                "FLD_ZONE": zone,
                "ZONE_SUBTY": subtype,
                "map_class": label,
                "area_acres": area,
                "pct_study_zone": 100 * area / (geometry.area / ACRE_M2),
            }
        )
detail_table = pd.DataFrame(detail_records)
detail_table.to_csv(FLOOD_DIR / "flood_zone_subtype_areas.csv", index=False)
display(detail_table)


## 11. Map floodway and flood-zone detail

Apply consistent colors and hatching to both the overview and parcel close-up.

In [ ]:
styles = {}
for label in sorted(flood_detail.map_class.unique()):
    rows = flood_detail.loc[flood_detail.map_class.eq(label)]
    zone = rows.FLD_ZONE.iloc[0]
    subtype = rows.ZONE_SUBTY.iloc[0]
    if "FLOODWAY" in subtype:
        styles[label] = {
            "facecolor": "#83b6d0",
            "edgecolor": "#164e72",
            "hatch": "////",
            "linewidth": 0.4,
        }
    elif subtype == "0.2 PCT ANNUAL CHANCE FLOOD HAZARD":
        styles[label] = {"facecolor": "#f1cd72", "edgecolor": "none"}
    elif subtype == "AREA OF MINIMAL FLOOD HAZARD":
        styles[label] = {"facecolor": "#e4e4e4", "edgecolor": "none"}
    elif subtype == "AREA WITH REDUCED FLOOD RISK DUE TO LEVEE":
        styles[label] = {"facecolor": "#c4b4db", "edgecolor": "none"}
    elif zone == "X":
        styles[label] = {
            "facecolor": "#cacaca",
            "edgecolor": "#777777",
            "hatch": "..",
            "linewidth": 0.3,
        }
    else:
        styles[label] = {"facecolor": "#c7e2ef", "edgecolor": "none"}
draw_detailed_flood_map()
draw_detailed_flood_map(parcel_view=True)


## 12. Record map categories

Append classification and styling details to the existing manifest.

In [ ]:
manifest_path = FLOOD_DIR / "manifest.json"
metadata = json.loads(manifest_path.read_text())
metadata["map_detail"] = {
    "fields": ["FLD_ZONE", "ZONE_SUBTY"],
    "classes": sorted(styles),
    "floodway_style": "diagonal hatch over blue fill",
    "detail_table": "flood_zone_subtype_areas.csv",
    "note": "No subtypes inferred where source fields are absent or unspecified; effective date unchanged.",
}
manifest_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
